# Habitat dan Migrasi Paus

**ID proyek:** `O005-LEGA-V101-PRJ11`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Bagaimana medan kesesuaian habitat sintetis dan kecenderungan migrasi menghasilkan lintasan populasi yang dapat divalidasi?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082211
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Paus bergerak pada bidang bujur–lintang sederhana menuju lintang habitat optimum yang berubah musiman, dengan variasi individu dan derau kecil.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
n_whales, n_days = 28, 120
longitude = rng.uniform(-24.0, 24.0, n_whales)
latitude = rng.normal(18.0, 3.0, n_whales)
tracks = np.empty((n_days + 1, n_whales, 2))
tracks[0, :, 0], tracks[0, :, 1] = longitude, latitude

def target_latitude(day):
    return 18.0 + 36.0 * (day / n_days)

for day in range(n_days):
    target = target_latitude(day + 1)
    latitude += 0.055 * (target - latitude) + rng.normal(0.0, 0.24, n_whales)
    longitude += -0.020 * longitude + rng.normal(0.0, 0.20, n_whales)
    latitude = np.clip(latitude, 5.0, 65.0)
    longitude = np.clip(longitude, -30.0, 30.0)
    tracks[day + 1, :, 0], tracks[day + 1, :, 1] = longitude, latitude

final_target = target_latitude(n_days)
initial_target_distance = np.abs(tracks[0, :, 1] - final_target)
final_target_distance = np.abs(tracks[-1, :, 1] - final_target)
lon_grid = np.linspace(-30.0, 30.0, 100)
lat_grid = np.linspace(5.0, 65.0, 100)
Lon, Lat = np.meshgrid(lon_grid, lat_grid)
habitat = np.exp(-0.5 * ((Lat - final_target) / 7.0) ** 2 - 0.5 * (Lon / 16.0) ** 2)


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert np.all((tracks[:, :, 0] >= -30.0) & (tracks[:, :, 0] <= 30.0))
assert np.all((tracks[:, :, 1] >= 5.0) & (tracks[:, :, 1] <= 65.0))
assert float(np.median(final_target_distance)) < 0.35 * float(np.median(initial_target_distance))
assert np.min(habitat) >= 0.0 and np.max(habitat) <= 1.0


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for whale in range(0, n_whales, 2):
    axes[0].plot(tracks[:, whale, 0], tracks[:, whale, 1], alpha=0.65, linewidth=1)
axes[0].axhline(final_target, color="black", linestyle="--", linewidth=1, label="optimum akhir")
axes[0].set(xlabel="bujur sintetis", ylabel="lintang sintetis", title="Lintasan migrasi")
axes[0].legend(fontsize=8)
image = axes[1].contourf(Lon, Lat, habitat, levels=16, cmap="viridis")
axes[1].scatter(tracks[-1, :, 0], tracks[-1, :, 1], s=12, color="white", edgecolor="black", linewidth=0.3)
axes[1].set(xlabel="bujur sintetis", ylabel="lintang sintetis", title="Kesesuaian habitat akhir")
fig.colorbar(image, ax=axes[1], label="kesesuaian")
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Peta bukan geografi nyata; tidak ada arus, batimetri, kapal, suara, struktur sosial, mortalitas, atau data telemetri sungguhan.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
